In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
feature_path = r"D:\NIDS\featured selected"

# Saari feature-selected files
parquet_files = [
    os.path.join(feature_path, file)
    for file in os.listdir(feature_path)
    if file.endswith(".parquet")
]

# LABEL ANALYSIS

all_labels = []

for file in parquet_files:

    df = pd.read_parquet(file)

    print("\n" + "=" * 70)
    print("FILE:", os.path.basename(file))
    print("=" * 70)

    # Label count
    print("\nLabel Distribution:")
    print(df["Label"].value_counts())

    # Percentage
    print("\nLabel Percentage:")
    print(
        (df["Label"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str) + "%"
    )

    # Unique labels
    print("\nUnique Labels:", df["Label"].nunique())
    print(df["Label"].unique())

    all_labels.extend(df["Label"].dropna().unique())


# ALL UNIQUE LABELS


print("\n" + "=" * 70)
print("ALL UNIQUE LABELS")
print("=" * 70)

unique_labels = sorted(set(all_labels))

print("Total Unique Labels:", len(unique_labels))

for label in unique_labels:
    print("-", label)


# ==============================
# BENIGN vs ATTACK
# ==============================

print("\n" + "=" * 70)
print("BENIGN vs ATTACK")
print("=" * 70)

total_benign = 0
total_attack = 0

for file in parquet_files:

    df = pd.read_parquet(file)

    benign_count = (
        df["Label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("benign")
        .sum()
    )

    attack_count = len(df) - benign_count

    total_benign += benign_count
    total_attack += attack_count

print("Total Benign:", total_benign)
print("Total Attack:", total_attack)

total = total_benign + total_attack

print(
    "Benign Percentage:",
    round((total_benign / total) * 100, 2), "%"
)

print(
    "Attack Percentage:",
    round((total_attack / total) * 100, 2), "%"
)


print("\n" + "=" * 70)
print("LABEL ANALYSIS COMPLETED")
print("=" * 70)


FILE: Benign-Monday-no-metadata.parquet

Label Distribution:
Label
Benign                        153677
Web Attack � Brute Force        1470
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Label Percentage:
Label
Benign                        98.62%
Web Attack � Brute Force       0.94%
Web Attack � XSS               0.42%
Web Attack � Sql Injection     0.01%
Name: proportion, dtype: str

Unique Labels: 4
<ArrowStringArray>
[                    'Benign',   'Web Attack � Brute Force',
           'Web Attack � XSS', 'Web Attack � Sql Injection']
Length: 4, dtype: str

FILE: Botnet-Friday-no-metadata.parquet

Label Distribution:
Label
Benign                        153677
Web Attack � Brute Force        1470
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Label Percentage:
Label
Benign                        98.62%
Web Attack � Brute Force       0.94%
Web Attack � XSS               0.4

## Encoding

In [3]:
df["Target"] = (df["Label"].str.strip().str.upper() != "BENIGN").astype(int)
print(df[['Label', 'Target']].head(10))

    Label  Target
0  Benign       0
1  Benign       0
2  Benign       0
3  Benign       0
4  Benign       0
5  Benign       0
6  Benign       0
7  Benign       0
8  Benign       0
9  Benign       0


In [4]:
print("Protocol data type:", df["Protocol"].dtype)
print("\nProtocol unique values:")
print(df["Protocol"].unique())

print("\nLabel data type:", df["Label"].dtype)
print("Target data type:", df["Target"].dtype)


Protocol data type: int8

Protocol unique values:
[ 6  0 17]

Label data type: str
Target data type: int64


## Train / Validation / Test Split

In [5]:
from glob import glob

files = glob(r"D:\NIDS\featured selected\*.parquet")

print("Total Files:", len(files))

for file in files:
    print(file)

Total Files: 8
D:\NIDS\featured selected\Benign-Monday-no-metadata.parquet
D:\NIDS\featured selected\Botnet-Friday-no-metadata.parquet
D:\NIDS\featured selected\Bruteforce-Tuesday-no-metadata.parquet
D:\NIDS\featured selected\DDoS-Friday-no-metadata.parquet
D:\NIDS\featured selected\DoS-Wednesday-no-metadata.parquet
D:\NIDS\featured selected\Infiltration-Thursday-no-metadata.parquet
D:\NIDS\featured selected\Portscan-Friday-no-metadata.parquet
D:\NIDS\featured selected\WebAttacks-Thursday-no-metadata.parquet


In [6]:
from glob import glob

featured_path = r"D:\NIDS\featured selected"

files = glob(os.path.join(featured_path, "*.parquet"))

for file in files:

    df = pd.read_parquet(file)

    # Target already hai to dobara nahi banayega
    if "Target" not in df.columns:

        df["Target"] = (
            df["Label"]
            .astype(str)
            .str.strip()
            .str.upper()
            .ne("BENIGN")
            .astype(int)
        )

        df.to_parquet(file, index=False)

        print("Target added:", os.path.basename(file))

    else:
        print("Target already exists:", os.path.basename(file))

print("\nTarget check completed.")

Target already exists: Benign-Monday-no-metadata.parquet
Target already exists: Botnet-Friday-no-metadata.parquet
Target already exists: Bruteforce-Tuesday-no-metadata.parquet
Target already exists: DDoS-Friday-no-metadata.parquet
Target already exists: DoS-Wednesday-no-metadata.parquet
Target already exists: Infiltration-Thursday-no-metadata.parquet
Target already exists: Portscan-Friday-no-metadata.parquet
Target already exists: WebAttacks-Thursday-no-metadata.parquet

Target check completed.


In [7]:
df.columns

Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
       'URG 

In [8]:
df = pd.read_parquet(files[0])

print(df.columns.tolist())
print("\nTarget distribution:")
print(df["Target"].value_counts())

['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Siz

In [9]:
import pandas as pd
import os
from glob import glob

# Featured-selected folder
featured_path = r"D:\NIDS\featured selected"

# Automatically find all parquet files
files = glob(os.path.join(featured_path, "*.parquet"))

print("Total Files:", len(files))

# Divide files by day

train_files = [
    f for f in files
    if any(day in os.path.basename(f)
           for day in ["Monday", "Tuesday", "Wednesday"])
]

val_files = [
    f for f in files
    if "Thursday" in os.path.basename(f)
]

test_files = [
    f for f in files
    if "Friday" in os.path.basename(f)
]

print("\nTraining Files:")
for f in train_files:
    print(os.path.basename(f))

print("\nValidation Files:")
for f in val_files:
    print(os.path.basename(f))

print("\nTest Files:")
for f in test_files:
    print(os.path.basename(f))

# Load files

train_df = pd.concat(
    [pd.read_parquet(f) for f in train_files],
    ignore_index=True
)

val_df = pd.concat(
    [pd.read_parquet(f) for f in val_files],
    ignore_index=True
)

test_df = pd.concat(
    [pd.read_parquet(f) for f in test_files],
    ignore_index=True
)









Total Files: 8

Training Files:
Benign-Monday-no-metadata.parquet
Bruteforce-Tuesday-no-metadata.parquet
DoS-Wednesday-no-metadata.parquet

Validation Files:
Infiltration-Thursday-no-metadata.parquet
WebAttacks-Thursday-no-metadata.parquet

Test Files:
Botnet-Friday-no-metadata.parquet
DDoS-Friday-no-metadata.parquet
Portscan-Friday-no-metadata.parquet


In [10]:
# Create X and y
drop_cols = ["Label", "Target", "Init Bwd Win Bytes", "Init Fwd Win Bytes"]
X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df["Target"]

X_val = val_df.drop(columns=drop_cols, errors='ignore')
y_val = val_df["Target"]

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df["Target"]

print("Artifact features dropped successfully!")

Artifact features dropped successfully!


In [11]:
# -----------------------------
# Check results
# -----------------------------

print("\n" + "=" * 70)
print("TRAIN / VALIDATION / TEST SPLIT COMPLETED")
print("=" * 70)

print("\nTraining:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTesting:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining Target:")
print(y_train.value_counts())

print("\nValidation Target:")
print(y_val.value_counts())

print("\nTesting Target:")
print(y_test.value_counts())


TRAIN / VALIDATION / TEST SPLIT COMPLETED

Training:
X_train: (467460, 65)
y_train: (467460,)

Validation:
X_val: (311640, 65)
y_val: (311640,)

Testing:
X_test: (467460, 65)
y_test: (467460,)

Training Target:
Target
0    461031
1      6429
Name: count, dtype: int64

Validation Target:
Target
0    307354
1      4286
Name: count, dtype: int64

Testing Target:
Target
0    461031
1      6429
Name: count, dtype: int64


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed successfully.")

print("\nShapes:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

Scaling completed successfully.

Shapes:
X_train_scaled: (467460, 65)
X_val_scaled: (311640, 65)
X_test_scaled: (467460, 65)


In [13]:
import joblib
os.makedirs(r"scaled",exist_ok=True)
np.save(os.path.join("scaled","X_train_scaled.npy"),X_train_scaled)
np.save(os.path.join("scaled","X_val_scaled.npy"),X_val_scaled)
np.save(os.path.join("scaled","X_test_scaled.npy"),X_test_scaled)
np.save(os.path.join("scaled", "y_train.npy"), y_train)
np.save(os.path.join("scaled", "y_val.npy"), y_val)
np.save(os.path.join("scaled", "y_test.npy"), y_test)
joblib.dump(scaler, os.path.join("scaled", "fitted_scaler.pkl"))

print("Scaled Data Save Successfully")

Scaled Data Save Successfully


In [14]:
##No Need Beacause I save Scaled data instead of train ,val and test data
# os.makedirs(r"split data", exist_ok=True)
# train_df.to_parquet(r"split data\train_combined.parquet", index=False)
# val_df.to_parquet(r"split data\val_combined.parquet", index=False)
# test_df.to_parquet(r"split data\test_combined.parquet", index=False)
